In [37]:
import pandas as pd
from functools import cache
from calitp_data_analysis.sql import query_sql
from calitp_data_analysis.gcs_pandas import GCSPandas

@cache
def gcs_pandas():
    return GCSPandas()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Get NTD ID from warehouse
for all California agences in NTD

In [2]:
# need to get ntd ID of all California agencies


# also include VOMs for comparison
ca_agencies = query_sql(
    """
    SELECT distinct 
        ntd_id, 
        agency, 
        agency_voms
    FROM cal-itp-data-infra.mart_ntd.dim_annual_service_agencies
    WHERE 
        state = 'CA' 
        AND report_year = 2024
    """,
    as_df=True,
)
ca_agencies = ca_agencies.add_suffix("_mart")
ca_agencies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ntd_id_mart       209 non-null    object 
 1   agency_mart       209 non-null    object 
 2   agency_voms_mart  209 non-null    float64
dtypes: float64(1), object(2)
memory usage: 5.0+ KB


# Read in NTD 2024 Revenue Vehicle Inventory
this list all the revenue vehicles reported by each agency

## NTD Glossary Definitions
- `Active Vehicles`
> The vehicles available to operate in revenue service at the end of the fiscal year, including: 
•   Spares
•   Vehicles temporarily out of service for routine maintenance and minor repairs
•   Operational vehicles

- Active Vehicles in Fleet
>The vehicles in a particular fleet at year-end that are available to operate in revenue service, including: 
•   Spares
•   Vehicles temporarily out of service for routine maintenance and minor repairs

- Number of Active Vehicles in Fleet
>The total number of operational revenue vehicles in the fleet available for general public transit service, including spare or back up revenue vehicles. The total should also include any operational revenue vehicles used by contractors in general public transit service. Non-revenue service vehicles and personal vehicles should not be included. Can be found in: A-30

- `Vehicles in Total Fleet`
>All revenue vehicles held at the end of the fiscal year, including those: 
•   In service;
•   In storage;
•   Emergency contingency; and
•   Awaiting sale.
Can be found in: A-30

- `Vehicles Operated in Annual Maximum Service (VOMS)`
>The number of revenue vehicles operated to meet the annual maximum service requirement. This is the revenue vehicle count during the peak season of the year; on the week and day that maximum service is provided. Vehicles operated in maximum service (VOMS) exclude: 
•   Atypical days; or
•   One-time special events.
Can be found in: B-30, A-30, S-10, Declarations, MR-20

In [3]:
# saved excel to GCS, so use gcs_pandas()
this_gcs_path = "gs://calitp-analytics-data/data-analyses/ntd/"

In [3]:
rev_veh_inv_link = f"{this_gcs_path}2024 Revenue Vehicle Inventory_250820.xlsx"

rev_veh = gcs_pandas().read_excel(rev_veh_inv_link)

rev_veh.columns = rev_veh.columns.str.lower().str.strip().str.replace(" ", "_")
rev_veh["ntd_id"] = rev_veh["ntd_id"].astype("str")
rev_veh.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35218 entries, 0 to 35217
Data columns (total 41 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   state/parent_ntd_id                           13899 non-null  object 
 1   ntd_id                                        35218 non-null  object 
 2   agency_name                                   35218 non-null  object 
 3   reporter_type                                 35218 non-null  object 
 4   reporting_module                              35218 non-null  object 
 5   group_plan_sponsor_ntdid                      18739 non-null  object 
 6   group_plan_sponsor_name                       18739 non-null  object 
 7   modes                                         35218 non-null  object 
 8   revenue_vehicle_inventory_id                  35218 non-null  int64  
 9   agency_fleet_id                               35218 non-null 

In [4]:
rev_veh.head(3)

,state/parent_ntd_id,ntd_id,agency_name,reporter_type,reporting_module,group_plan_sponsor_ntdid,group_plan_sponsor_name,modes,revenue_vehicle_inventory_id,agency_fleet_id,modetos_vehicles_operated_in_maximum_service,total_fleet_vehicles,dedicated_fleet,vehicle_type,ownership_type,funding_source,manufacture_year,rebuild_year,type_of_last_renewal,useful_life_benchmark,manufacturer,other_manufacturer_description,model,active_fleet_vehicles,ada_fleet_vehicles,emergency_contingency_vehicles,fuel_type,other_fuel_description,vehicle_length,seating_capacity,standing_capacity,total_miles_on_active_vehicles_during_period,average_lifetime_miles_per_active_vehicles,no_capital_replacement_flag,separate_asset_flag,event_data_recorders,emergency_lighting_system_design,emergency_signage,emergency_path_marking,automated_vehicles_flag,notes
0,NaN,1,King County,Full Reporter,Urban,NaN,NaN,DR/PT,37760,,366.0,2,Y,Cutaway,Owned outright by public agency (OOPA),Non-Federal Public Funds (NFPA),2009.0,NaN,NaN,10.0,Startrans (Supreme Corporation),NaN,SupremeSenator,0,0.0,0.0,Diesel Fuel,NaN,22.0,13,0.0,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,1,King County,Full Reporter,Urban,NaN,NaN,DR/PT,42645,,366.0,14,Y,Cutaway,Owned outright by public agency (OOPA),Non-Federal Public Funds (NFPA),2010.0,NaN,NaN,10.0,Startrans (Supreme Corporation),NaN,SupremeSenator,14,14.0,0.0,Diesel Fuel,NaN,22.0,13,0.0,66135.0,302864.0,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1,King County,Full Reporter,Urban,NaN,NaN,DR/PT,48057,,366.0,8,Y,Cutaway,Owned outright by public agency (OOPA),Non-Federal Public Funds (NFPA),2011.0,NaN,NaN,10.0,Startrans (Supreme Corporation),NaN,SupremeSenator,8,8.0,0.0,Gasoline,NaN,22.0,13,0.0,80753.0,220649.0,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# filter the NTD list by the NTD id from the warehouse

ca_rev_veh = rev_veh[
    rev_veh["ntd_id"].isin(ca_agencies["ntd_id_mart"].unique().tolist())
]

In [7]:
# group by / aggregate by active fleet, total fleet, VOMS.
# maybe there is a difference between all of the. I expect the toal fleet to be the highest

ca_rev_veh = (
    ca_rev_veh.groupby(
        [
            "ntd_id",
            "agency_name",
        ]
    )
    .agg(
        {
            "total_fleet_vehicles": "sum",
            "active_fleet_vehicles": "sum",
            "modetos_vehicles_operated_in_maximum_service": "max",
        }
    )
    .reset_index()
)

In [8]:
ca_rev_veh.head()

,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service
0,90003,San Francisco Bay Area Rapid Transit District,785,776,566.0
1,90004,Golden Empire Transit District,160,160,51.0
2,90006,Santa Cruz Metropolitan Transit District,147,147,58.0
3,90008,City of Santa Monica,239,239,124.0
4,90009,San Mateo County Transit District,468,468,178.0


## difference between VOMs and Rev Vehicle Inv.

In [9]:
diff_voms_ref_veh = ca_rev_veh.merge(
    ca_agencies,
    left_on="ntd_id",
    right_on="ntd_id_mart",
    how="inner",
    # suffixes = ("_ntd_excel","_warehouse"),
    # indicator=True # dont really need this since how=inner
)

In [10]:
diff_voms_ref_veh.head()

,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,ntd_id_mart,agency_mart,agency_voms_mart
0,90003,San Francisco Bay Area Rapid Transit District,785,776,566.0,90003,"San Francisco Bay Area Rapid Transit District,...",582.0
1,90004,Golden Empire Transit District,160,160,51.0,90004,Golden Empire Transit District,93.0
2,90006,Santa Cruz Metropolitan Transit District,147,147,58.0,90006,Santa Cruz Metropolitan Transit District,97.0
3,90008,City of Santa Monica,239,239,124.0,90008,"City of Santa Monica, dba: Big Blue Bus",162.0
4,90009,San Mateo County Transit District,468,468,178.0,90009,"San Mateo County Transit District, dba: SamTrans",361.0


## Calculate the differencs in VOMS values between the revenue vehicle inventory and the warehouse
Note, the warehouse table pull from a NTD excel file as well, so VOMS should be the same...

In [11]:
diff_voms_ref_veh["voms_diff"] = (
    diff_voms_ref_veh["agency_voms_mart"]
    - diff_voms_ref_veh["modetos_vehicles_operated_in_maximum_service"]
)

In [12]:
diff_voms_ref_veh["total_fleet_veh_voms_diff"] = (
    diff_voms_ref_veh["total_fleet_vehicles"]
    - diff_voms_ref_veh["modetos_vehicles_operated_in_maximum_service"]
)

In [17]:
diff_voms_ref_veh["ntd_id"].nunique()

209

## save data to GCS

In [14]:
# diff_voms_ref_veh.to_csv(f"{this_gcs_path}1943_total_revenue_vehicles.csv")

## read in GCS data

In [4]:
diff_voms_ref_veh = gcs_pandas().read_csv(
    f"{this_gcs_path}1943_total_revenue_vehicles.csv"
)

In [6]:
diff_voms_ref_veh.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 11 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Unnamed: 0                                    209 non-null    int64  
 1   ntd_id                                        209 non-null    int64  
 2   agency_name                                   209 non-null    object 
 3   total_fleet_vehicles                          209 non-null    int64  
 4   active_fleet_vehicles                         209 non-null    int64  
 5   modetos_vehicles_operated_in_maximum_service  209 non-null    float64
 6   ntd_id_mart                                   209 non-null    int64  
 7   agency_mart                                   209 non-null    object 
 8   agency_voms_mart                              209 non-null    float64
 9   voms_diff                                     209 non-null    flo

# Count of routes

Eric used one of the new roll up tables using this query
```
SELECT DISTINCT month, month_first_day, schedule_name, n_routes, day_type
FROM `cal-itp-data-infra.mart_gtfs_rollup.fct_monthly_operator_summary`
WHERE month_first_day = '2026-02-01' AND n_routes > 0 AND day_type = 'Weekday'
ORDER BY n_routes DESC
```

In [5]:
rollup_routes = gcs_pandas().read_csv(
    f"{this_gcs_path}agency_routes_02-2026_rollup_routes_results.csv"
)

In [7]:
rollup_routes.info() # this has shcedule name but not analysis name. may need the custom query to join the bridge table

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239 entries, 0 to 238
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   month            239 non-null    int64 
 1   month_first_day  239 non-null    object
 2   schedule_name    239 non-null    object
 3   n_routes         239 non-null    int64 
 4   day_type         239 non-null    object
dtypes: int64(2), object(3)
memory usage: 9.5+ KB


In [49]:
query="""
WITH
  t1 AS (
    SELECT *
    FROM
      cal-itp-data-infra.mart_transit_database.bridge_gtfs_analysis_name_x_ntd
  ),
  t2 AS (
    SELECT DISTINCT month, month_first_day, schedule_name, n_routes, day_type
    FROM `cal-itp-data-infra.mart_gtfs_rollup.fct_monthly_operator_summary`
    WHERE
      month_first_day = '2026-02-01' AND n_routes > 0 AND day_type = 'Weekday'
    ORDER BY n_routes DESC
  )
SELECT DISTINCT
  t1.schedule_gtfs_dataset_name,
  t1.analysis_name,
  t1.ntd_id_2022,
  t2.schedule_name,
  t2.n_routes
FROM
  t1
inner JOIN t2
  ON t1.schedule_gtfs_dataset_name = t2.schedule_name;
"""
rollup_route_bridge = query_sql(
    query,
    as_df=True
)

rollup_route_bridge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   schedule_gtfs_dataset_name  197 non-null    object
 1   analysis_name               197 non-null    object
 2   ntd_id_2022                 163 non-null    object
 3   schedule_name               197 non-null    object
 4   n_routes                    197 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 7.8+ KB


In [50]:
rollup_route_bridge.sample(3)

,schedule_gtfs_dataset_name,analysis_name,ntd_id_2022,schedule_name,n_routes
32,Bay Area 511 Santa Clara Transit Schedule,Santa Clara Valley Transportation Authority,90013,Bay Area 511 Santa Clara Transit Schedule,67
183,Placer Schedule,Placer County,90196,Placer Schedule,7
123,Yurok Schedule,Yurok Tribe,99262,Yurok Schedule,1


# Agencies without GTFS-RT
via CS team

In [51]:
no_rt = gcs_pandas().read_excel(
    f"{this_gcs_path}organizations_without_gtfs_rt_data_by_total_voms_2026-02-23_CS additions 1 (003).xlsx"
)

no_rt.columns = no_rt.columns.str.lower().str.strip().str.replace(" ", "_")

no_rt = no_rt[(no_rt["organization_name"].notna()) & (no_rt["gtfs_rt"].str.contains("No"))]

no_rt.sample(3)

,organization_name,service_name,total_2023_voms,is_the_agency_pursuing_rt_(via_the_msas_or_not)?,notes,passiogo,gtfs_rt,if_gtfs-rt_data_-_add_the_link
4,Palos Verdes Peninsula Transit Authority,Palos Verdes Peninsula Transit Authority,27.0,NaN,NaN,NaN,No,NaN
12,El Dorado County Transit Authority,El Dorado Transit,17.0,Maybe,Expressed interest in RT through MSAs,NaN,No,NaN
1,Yuma County Intergovernmental Public Transport...,Yuma County Area Transit,54.0,NaN,NaN,NaN,No,NaN


# Merge
1. `diff_voms_ref_veh` - to - `no_rt`
2. `rollup_routes` - to - `no_rt`

Hopefully the names are consistent between the datasets

In [54]:
no_rt_keep_cols = [
    'organization_name', 
    'service_name', 
    # 'total_2023_voms',
    'is_the_agency_pursuing_rt_(via_the_msas_or_not)?', 
    # 'notes', 
    # 'passiogo',
    'gtfs_rt', 
    # 'if_gtfs-rt_data_-_add_the_link'
]
no_rt_rev_veh_inv = diff_voms_ref_veh.merge(
    no_rt[no_rt_keep_cols],
    left_on = "agency_name",
    right_on = "organization_name",
    how = "inner",
)

no_rt_rev_veh_inv

,Unnamed: 0,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,ntd_id_mart,agency_mart,agency_voms_mart,voms_diff,total_fleet_veh_voms_diff,organization_name,service_name,is_the_agency_pursuing_rt_(via_the_msas_or_not)?,gtfs_rt
0,30,90052,City of Corona,20,20,6.0,90052,City of Corona,11.0,5.0,14.0,City of Corona,Corona Cruiser,Maybe,No
1,45,90119,City of Laguna Beach,32,32,18.0,90119,"City of Laguna Beach, dba: Laguna Beach Transit",24.0,6.0,14.0,City of Laguna Beach,Laguna Beach Trolley,NaN,No
2,53,90149,City of Lompoc,16,16,2.0,90149,"City of Lompoc, dba: City of Lompoc Transit",10.0,8.0,14.0,City of Lompoc,City of Lompoc Transit,NaN,No
3,69,90175,City of Lodi,25,25,8.0,90175,"City of Lodi, dba: GrapeLine",13.0,5.0,17.0,City of Lodi,Grapeline,NaN,No
4,74,90199,City of Madera,22,22,14.0,90199,"City of Madera, dba: Madera Metro",22.0,8.0,8.0,City of Madera,Madera Metro,NaN,No
5,87,90226,Imperial County Transportation Commission,64,64,20.0,90226,Imperial County Transportation Commission,38.0,18.0,44.0,Imperial County Transportation Commission,Imperial Valley Transit,Yes,No
6,89,90229,El Dorado County Transit Authority,50,50,7.0,90229,"El Dorado County Transit Authority, dba: El Do...",18.0,11.0,43.0,El Dorado County Transit Authority,El Dorado Transit,Maybe,No
7,93,90238,City of Delano,17,17,5.0,90238,"City of Delano, dba: Delano Area Rapid Transit",9.0,4.0,12.0,City of Delano,Delano Area Rapid Transit,NaN,No
8,100,90252,City of Bell,23,22,8.0,90252,City of Bell,10.0,2.0,15.0,City of Bell,La Campana,NaN,No
9,101,90253,City of Bell Gardens,7,7,3.0,90253,City of Bell Gardens,6.0,3.0,4.0,City of Bell Gardens,Bell Gardens Trolley,NaN,No


In [55]:
no_rt_rollup_routes = rollup_route_bridge.merge(
    no_rt[no_rt_keep_cols],
    left_on = "analysis_name",
    right_on = "organization_name",
    how = "inner"
)

In [56]:
display(
    no_rt_rollup_routes.info(),
    no_rt_rollup_routes
)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 9 columns):
 #   Column                                            Non-Null Count  Dtype 
---  ------                                            --------------  ----- 
 0   schedule_gtfs_dataset_name                        46 non-null     object
 1   analysis_name                                     46 non-null     object
 2   ntd_id_2022                                       46 non-null     object
 3   schedule_name                                     46 non-null     object
 4   n_routes                                          46 non-null     int64 
 5   organization_name                                 46 non-null     object
 6   service_name                                      46 non-null     object
 7   is_the_agency_pursuing_rt_(via_the_msas_or_not)?  10 non-null     object
 8   gtfs_rt                                           46 non-null     object
dtypes: int64(1), object(8)
memory usage

None

,schedule_gtfs_dataset_name,analysis_name,ntd_id_2022,schedule_name,n_routes,organization_name,service_name,is_the_agency_pursuing_rt_(via_the_msas_or_not)?,gtfs_rt
0,Delano Schedule,City of Delano,90238,Delano Schedule,4,City of Delano,Delano Area Rapid Transit,NaN,No
1,El Dorado Schedule,El Dorado County Transit Authority,90229,El Dorado Schedule,8,El Dorado County Transit Authority,El Dorado Transit,Maybe,No
2,Lompoc Schedule,City of Lompoc,90149,Lompoc Schedule,6,City of Lompoc,City of Lompoc Transit,NaN,No
3,Glenn Schedule,Glenn County,91088,Glenn Schedule,1,Glenn County,Glenn Ride,NaN,No
4,County Express Schedule,San Benito County Local Transportation Authority,91009,County Express Schedule,1,San Benito County Local Transportation Authority,County Express,NaN,No
5,Get Around Town Express Schedule,City of South Gate,90291,Get Around Town Express Schedule,2,City of South Gate,Get Around Town Express,NaN,No
6,Amador Schedule,Amador Regional Transit System,91000,Amador Schedule,5,Amador Regional Transit System,Amador Transit,NaN,No
7,LADPW Schedule,Los Angeles County,90271,LADPW Schedule,18,Los Angeles County,Los Angeles County Transit Services,NaN,No
8,Siskiyou Schedule,Siskiyou County,91048,Siskiyou Schedule,6,Siskiyou County,Siskiyou Transit and General Express,Maybe,No
9,El Monte RTAP Schedule,City of El Monte,90265,El Monte RTAP Schedule,10,City of El Monte,El Monte Shuttles and Trolleys,NaN,No
